# ToyAIKit: From Your Agent Loop to a Framework

This notebook translates the manual agentic loop into ToyAIKit in an explanation-first style. Each code cell now uses the same four comment sections: **Before this line**, **At this line**, **After this line**, and **Possible output**. [cite:322]


## What ToyAIKit is

ToyAIKit is presented as a minimal educational framework for LLM assistants with tool use. Its repository describes it as useful for learning, prototyping, and debugging, while recommending more production-ready frameworks such as PydanticAI or the OpenAI Agents SDK once you are comfortable with the basics. [page:1]


## Translation table

| Manual concept | ToyAIKit equivalent | Meaning |
|---|---|---|
| `make_call(...)` plus tool routing | `Tools()` | Registers tools and handles dispatching |
| `responses.create(...)` setup | `OpenAIClient` | Wraps model and SDK client |
| Inner tool-call loop | `OpenAIResponsesRunner.loop(...)` | Handles repeated tool-calling iterations |
| Outer Q&A loop | `OpenAIResponsesRunner.run()` | Handles the interactive chat session |
| `message_history` | `result.all_messages` / `previous_messages` | Keeps conversation memory |
| Debug prints | `IPythonChatInterface` + callback | Shows what happens during execution |

ToyAIKit keeps the same core pattern you learned manually: the LLM can decide to call tools or answer, and the framework manages the repeated loop until the response is complete. [page:1]


## Dependency diagram

```text
search(query)          add_entry(filename, title, description, content)
      |                               |
      +---------------+---------------+
                      v
               agent_tools = Tools()
               add_tool(function, schema)
                      |
                      v
           OpenAIResponsesRunner
           - tools
           - developer_prompt
           - llm_client
                      |
                      v
               agent.loop(prompt)
                      |
                      v
          framework runs inner loop for you
            LLM needs a tool?
              |          |
             yes         no
              |          |
              v          v
      matching Python    final answer
      function executes  returned
              |
              v
      tool result goes back to LLM
```

The important shift is that you no longer write the routing loop yourself. The framework owns that plumbing, but your Python functions still do the real work. [page:1]


## Install

ToyAIKit can be installed with `pip install toyaikit`, and your environment may also use `uv add toyaikit` if your project is managed by `uv`. The ToyAIKit repository explicitly documents `pip install toyaikit` as the quick-start install command. [page:1]


In [7]:
# Terminal option:
! uv add toyaikit
#
# Notebook option:
# %pip install -q toyaikit gitsource minsearch openai

# Before this line:
# - Required packages may not yet exist in the current notebook environment.
# - The kernel may not know about ToyAIKit or the supporting libraries.
#
# At this line:
# - We install ToyAIKit and the supporting packages needed for this chapter.
#
# After this line:
# - The environment should contain `toyaikit`, `gitsource`, `minsearch`, and `openai`.
# - You may still need to restart the kernel if the package was newly installed.
#
# Possible output:
# - Installation logs
# - Or a message saying the packages are already installed


Resolved 141 packages in 1ms
Checked 137 packages in 2ms


## Build the knowledge base

The framework handles the agent loop, but your application still owns the data setup. We keep using the Evidently documentation index from the earlier lessons.


In [8]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner='evidentlyai',
    repo_name='docs',
    allowed_extensions={'md', 'mdx'},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=['title', 'description', 'content'],
    keyword_fields=['filename'],
)
index.fit(chunked_docs)

print('Files:', len(files))
print('Chunks:', len(chunked_docs))

# Before this line:
# - We do not yet have a searchable documentation index.
# - The notebook only has imports available.
#
# At this line:
# - We read the Evidently docs repository, chunk the documents, and fit a searchable index.
#
# After this line:
# - `index` contains searchable documentation chunks.
# - The later `search(...)` tool can use this index.
#
# Possible output:
# - Files: <positive number>
# - Chunks: <positive number>


Files: 95
Chunks: 382


## Define real Python functions

These are still normal Python functions. ToyAIKit will register and call them, but it does not replace your actual business logic.


In [9]:
def search(query):
    return index.search(query=query, num_results=5)


def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return 'OK'

# Before this line:
# - We have a searchable index, but no callable tool functions yet.
#
# At this line:
# - We define the real Python functions that the framework can later expose as tools.
#
# After this line:
# - `search` can query the index.
# - `add_entry` can append a new item into the index.
#
# Possible output:
# - No printed output


## Define tool schemas

A tool schema tells the model the tool name, purpose, and parameters. The schema is metadata; it is not the real Python function.


In [10]:
search_tool = {
    'type': 'function',
    'name': 'search',
    'description': 'Search the documentation database for relevant results based on a query string.',
    'parameters': {
        'type': 'object',
        'properties': {
            'query': {'type': 'string'}
        },
        'required': ['query'],
    },
}

add_entry_tool = {
    'type': 'function',
    'name': 'add_entry',
    'description': 'Add a new documentation entry to the index.',
    'parameters': {
        'type': 'object',
        'properties': {
            'filename': {'type': 'string'},
            'title': {'type': 'string'},
            'description': {'type': 'string'},
            'content': {'type': 'string'},
        },
        'required': ['filename', 'title', 'description', 'content'],
    },
}

# Before this line:
# - The model does not yet know which tools are available or what arguments they accept.
#
# At this line:
# - We define the tool metadata that ToyAIKit can send to the model.
#
# After this line:
# - `search_tool` and `add_entry_tool` describe the tool interface.
# - They still need to be registered with the real Python functions.
#
# Possible output:
# - No printed output


## Register tools with ToyAIKit

This is the framework version of the manual `make_call(...)` binding step. The tools system in ToyAIKit is designed to integrate Python functions with model function calling. [page:1]


In [11]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.add_tool(add_entry, add_entry_tool)

agent_tools.get_tools()

# Before this line:
# - We have function definitions and tool schemas, but nothing connects them yet.
#
# At this line:
# - We register each real Python function together with its tool schema inside a ToyAIKit `Tools` object.
#
# After this line:
# - `agent_tools` now knows how to expose and route the `search` and `add_entry` tools.
# - `get_tools()` returns the definitions that will be sent to the LLM.
#
# Possible output:
# - A list of tool definitions including 'search' and 'add_entry'


[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string'}},
   'required': ['query']}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string'},
    'title': {'type': 'string'},
    'description': {'type': 'string'},
    'content': {'type': 'string'}},
   'required': ['filename', 'title', 'description', 'content']}}]

## Instructions

The prompt still matters. The framework automates the loop, but the instructions still guide the model’s decisions.


In [12]:
instructions = """
You're a documentation assistant.

Answer the user question using the documentation knowledge base.
If you cannot find the answer, say so.

When research is needed:
1. Start with one focused search.
2. Analyze the results and perform targeted follow-up searches if needed.
3. Synthesize the evidence into the final answer.
""".strip()

# Before this line:
# - The framework objects exist, but we have not yet defined the agent's behavior.
#
# At this line:
# - We define the developer prompt that tells the model how to behave.
#
# After this line:
# - `instructions` contains the system guidance the runner will use.
#
# Possible output:
# - No printed output


## Create the LLM client

`OpenAIClient` packages the model name and OpenAI SDK client so you do not keep rewriting that setup. The repository shows this pattern in its OpenAI examples. [page:1]


In [13]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient

llm_client = OpenAIClient(
    model='gpt-4o-mini',
    client=OpenAI(),
)

# Before this line:
# - We have tools and instructions, but nothing yet that can talk to the model.
#
# At this line:
# - We create the client wrapper that ToyAIKit will use to call the OpenAI Responses API.
#
# After this line:
# - `llm_client` stores the model and the SDK client.
# - The runner can now use it for model calls.
#
# Possible output:
# - No printed output


## Create the runner

ToyAIKit calls agents **runners**. `OpenAIResponsesRunner` packages tools, prompt, and model client into one object. [page:1]


In [14]:
from toyaikit.chat.runners import OpenAIResponsesRunner

agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=llm_client,
)

# Before this line:
# - We have all the ingredients, but not yet a single object that can run the agentic loop.
#
# At this line:
# - We create a ToyAIKit runner with tools, prompt, and LLM client.
#
# After this line:
# - `agent` can now execute the inner tool-call loop through `agent.loop(...)`.
#
# Possible output:
# - No printed output


## Run one inner loop

This is the framework equivalent of the manual inner loop. The runner can make multiple model/tool iterations before returning the result.


In [15]:
result = agent.loop('how can I create an Evidently dashboard?')

print(result.last_message)
print(result.cost.total_cost)

# Before this line:
# - The runner exists, but no question has been asked yet.
#
# At this line:
# - We ask one question and let ToyAIKit manage the full inner tool-call loop.
#
# After this line:
# - `result` contains the final answer, message history, and cost details.
#
# Possible output:
# - <final assistant answer>
# - <numeric cost value>


To create an **Evidently dashboard**, follow these steps:

### Step 1: Create a Project
You need to create a project and run an evaluation at least once. This evaluates your AI application and saves a Report to the Project, which will serve as the data source for your Dashboard.

### Step 2: Access the Dashboard
- **Evidently OSS**: Use the Python API to create and manage dashboards.
- **Evidently Cloud and Enterprise**: Use the user interface (UI) to create dashboards without coding.

### Step 3: Add Dashboard Tabs
You can organize your Dashboard into multiple tabs:
- In the UI, enter **Edit mode** and click the plus sign with "add Tab."
- In the API, use the following command:
    ```python
    project.dashboard.add_tab("Tab Name")
    ```

### Step 4: Add Panels
Each Panel displays visual elements like counters or plots based on selected metrics:
- **Using the API**:
    ```python
    from evidently.sdk.models import PanelMetric
    from evidently.sdk.panels import DashboardPanelPlo

## Add visibility with a callback

Without visibility, the framework can feel like a black box. The notebook interface and callback let you see the function calls and responses as they happen. The ToyAIKit repository documents an IPython chat interface that can display user input, assistant responses, and function calls. [page:1]


In [16]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

result = agent.loop(
    'how can I create an Evidently dashboard?',
    callback=runner_callback,
)

# Before this line:
# - The runner can work, but you cannot yet observe each internal step.
#
# At this line:
# - We create a notebook chat interface and a display callback, then run the loop with visibility enabled.
#
# After this line:
# - The loop shows model messages, tool calls, and the final answer as they happen.
#
# Possible output:
# - Visible search(...) calls
# - Visible assistant messages
# - Visible final answer


-> Response received


-> Response received


## Continue the conversation

Passing `previous_messages=result.all_messages` is the framework version of reusing `message_history` in the next request.


In [17]:
result2 = agent.loop(
    'show me the code',
    callback=runner_callback,
    previous_messages=result.all_messages,
)

print(result2.last_message)

# Before this line:
# - We have a previous result, but the next loop does not yet know about it.
#
# At this line:
# - We run another loop and explicitly pass the earlier conversation history into `previous_messages`.
#
# After this line:
# - `result2` can use the prior context instead of starting from scratch.
#
# Possible output:
# - A follow-up answer that may reuse earlier context without another search


-> Response received


Here’s a sample code snippet to create a dashboard and add panels using the Python API in Evidently:

### Example Code for Creating a Dashboard and Adding Panels

```python
from evidently.sdk.models import PanelMetric
from evidently.sdk.panels import DashboardPanelPlot
from evidently.sdk import Project

# Connect to Evidently and create a project (replace with your credentials)
token = "YOUR_API_KEY"
project_name = "MyProject"

# Initialize your project
project = Project(token=token, name=project_name)

# Add a new Dashboard Tab
project.dashboard.add_tab("Performance Metrics")

# Add a Counter Panel displaying Row Count
project.dashboard.add_panel(
    DashboardPanelPlot(
        title="Total Evaluations",
        subtitle="The total number of evaluations over time.",
        size="half",
        values=[PanelMetric(legend="Row count", metric="RowCount")],
        plot_params={"plot_type": "counter"}
    ),
    tab="Performance Metrics"
)

# Add a Line Plot Panel showing Accuracy
proje

## Run the outer Q&A loop

`run()` is the framework equivalent of the manual outer `while True: input(...)` loop. It provides an interactive chat session. [page:1]


In [18]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client,
)

result = agent.run()

# Before this line:
# - We only have single-turn loop execution so far.
#
# At this line:
# - We create an interactive runner and start the outer Q&A loop.
#
# After this line:
# - The agent can keep chatting across multiple user turns until the session ends.
#
# Possible output:
# - You: <your question>
# - Visible function calls and answers
# - The session continues until stopped


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


## Manual code vs ToyAIKit

```text
Manual version:
responses.create(...)
  -> inspect output
  -> route tool call
  -> run function
  -> append result
  -> repeat

ToyAIKit version:
agent.loop(prompt)
  -> runner calls model
  -> framework routes tool call
  -> framework executes function
  -> framework stores result
  -> repeat
```

ToyAIKit does not remove the core ideas. It packages the loop, registration, routing, message handling, visibility, and cost tracking into reusable pieces. [page:1]


## Debugging checklist

- If `toyaikit` does not import, confirm it is installed in the notebook kernel environment.
- If a tool never runs, verify `agent_tools.add_tool(...)` executed successfully.
- If the model forgets earlier context, pass `previous_messages=result.all_messages`.
- If the runner feels opaque, add `DisplayingRunnerCallback`.
- If you changed a function in a notebook, rerun that cell and recreate the runner.

ToyAIKit is useful for experiments and learning, but the project does not position it as a production-grade framework. [page:1]


## Practice tasks

1. Add a new tool such as `count_results(query)`.
2. Compare a follow-up question with and without `previous_messages`.
3. Observe the callback output and write down the exact step order.
4. Add input validation inside `search(...)`.
5. Later, rebuild the same idea in PydanticAI and map the equivalent pieces.
